# 莫尔斯势与断键

In [ ]:
import micropip
await micropip.install(['matplotlib', 'numpy', 'pint'])

## 实验逻辑链条

**大前提（宏观热力学定律）**：根据热力学常识，化学键的断裂必然是一个吸热过程 ($\Delta H > 0$)。

**小前提（微观动力学图像）**：在莫尔斯势能井中，系统吸收的热能会转化为原子的动能。由于振动是往复的，这笔动能不仅可以驱动原子向右远离（拉伸 $r>r_e$），也同样可以驱动原子向左靠近（压缩 $r<r_e$）。

**现象与矛盾（提出问题）**：宏观上，化学键断裂的唯一定义是两个原子彻底分离（即微观图像上必须满足 $r\to \infty$， 向右走）。那么，如果系统吸收了足以断键的巨大能量，但原子的初始运动方向却偏偏向左（相互靠近）的，它们会“撞死”在势能井的左侧吗？吸热还能导致断键吗？

**核心假说（The Hypothesis）**：即使在最极端的初始条件下——即吸收的热能（$E_k \ge D_e$）在 $t=0$ 时全部转化为**向左的动能**——由于莫尔斯势能曲线极度不对称（左侧核斥力壁极其陡峭且近乎无限高，右侧引力坡平缓且高度有限），系统在极短时间内撞击左侧斥力壁后，必然发生**完全弹性反弹**。这股巨大的能量最终将被导向右侧，冲破解离能 ($D_e$) 的束缚，完成断键。

**实验验证目标**：通过建立一维分子动力学模型，设置初始位置为 $r_e$，初始动能 $E_k > D_e$，初始速度 $v < 0$（向左），追踪 $r(t)$ 的轨迹，观察其最终是否满足 $r(t) \to \infty$。

## 知识准备

### 莫尔斯势

$$
V(r) = D_e \left[ 1 - e^{-a(r-r_e)} \right]^2
$$

其中：

- $r$ (Internuclear Distance, 核间距): 系统的唯一动态变量，代表两个原子核之间的实时距离。
- $r_e$ (Equilibrium bond distance, 平衡键长): 势能达到最低点时的距离，即宏观测得的化学键平均长度。
- $D_e$ (Well depth / Dissociation energy, 势阱深度/经典解离能): 将原子从平衡位置拉到无穷远所需输入的**总经典能量**。（与量子力学中的真实键能 $D_0$ 作区分）。
- $a$ (Width parameter, 宽度参数/形态参数): 决定势阱的陡峭程度。

我们约定 $V(r_e) = 0$，即当系统处于平衡位置时，势能为零。这样能保证我们算出来的势能总是一个正数，在动力学计算中更为直观。

In [ ]:
# 绘制莫尔斯势能井图

import numpy as np
import matplotlib.pyplot as plt
import pint

ureg = pint.UnitRegistry()
Q_ = ureg.Quantity

# 定义莫尔斯势
def morse_potential(r, De, re, a):
    return De * (1 - np.exp(-a * (r - re)))**2

# 定义 H2 分子的物理参数
De = Q_(4.75, 'eV')
re = Q_(0.74, 'angstrom')
a = Q_(1.93, 'angstrom**-1')

# ================= 画图代码 =================

# 1. 生成距离数组 r (从极度压缩的 0.3 Å 到彻底拉断的 4.0 Å)
# 注意：赋予它 angstrom 单位，以便传入具有量纲的函数中
r_array = Q_(np.linspace(0.3, 4.0, 500), 'angstrom')

# 2. 计算对应的势能 V(r)
# 此时传出的 V_array 会自动带有 eV 单位 (因为 pint 处理了无量纲的指数部分)
V_array = morse_potential(r_array, De, re, a)

# 3. 剥离单位，提取纯数值用于 matplotlib 画图
r_plot = r_array.to('angstrom').magnitude
V_plot = V_array.to('eV').magnitude

# 4. 开始绘制极具物理美感的图表
plt.figure(figsize=(10, 6))

# 画出主曲线
plt.plot(r_plot, V_plot, color='blue', linewidth=2, label='Morse Potential $V(r)$')

# 添加物理学辅助线
plt.axhline(y=De.magnitude, color='red', linestyle='--', alpha=0.7, 
            label=f'Dissociation Energy $D_e$ ({De.magnitude} eV)')
plt.axvline(x=re.magnitude, color='green', linestyle='--', alpha=0.7, 
            label=f'Equilibrium Bond Length $r_e$ ({re.magnitude} Å)')
plt.axhline(y=0, color='black', linewidth=1) # 势能零点基准线

# 坐标轴与标题设置
plt.title('Morse Potential Energy Well of $H_2$ Molecule', fontsize=16, pad=15)
plt.xlabel('Internuclear Distance $r$ (Å)', fontsize=14)
plt.ylabel('Potential Energy $V(r)$ (eV)', fontsize=14)

# 【关键设置】限制 Y 轴高度，防止左侧排斥墙数值爆炸导致图表比例失调
plt.ylim(-0.5, 8.0) 
plt.xlim(0.2, 4.0)

# 其他美化
plt.legend(fontsize=12, loc='upper right')
plt.grid(True, linestyle=':', alpha=0.6)

# 显示图像
plt.tight_layout()
plt.show()

### 折合质量

**两体问题的单体化等效 (Two-Body Problem Reduction)**

假设我们有一根数轴。氢原子 1 在坐标 $x_1$，氢原子 2 在坐标 $x_2$。

为了方便，假设 $x_2 > x_1$。那么它们之间的键长（相对距离）就是：

$$
r = x_2 - x_1
$$

莫尔斯势的力 $\displaystyle F(r) = -\frac{dV}{dr}$，这是它们之间的相互作用的力。

根据牛顿第三定律（作用力与反作用力），这两个原子受到的力大小相等、方向相反：

- 原子 2 受到的力：向左拉（或向右推），加速度由 $m_2$ 决定。

$$
m_2 \ddot{x}_2 = F(r)
$$

- 原子 1 受到的力：方向刚好相反，所以加个负号。

$$
m_1 \ddot{x}_1 = -F(r)
$$

在化学键断裂的实验里，我们完全不关心这个分子是整体往左飘还是往右飘（即质心的运动），我们只关心俩原子之间距离 $r$ 的变化。

所以，我们去求距离 $r$ 的加速度 $\ddot{r}$。

因为 $r = x_2 - x_1$，所以它的加速度也就是两者的加速度之差：

$$
\ddot{r} = \ddot{x}_2 - \ddot{x}_1
$$

现在，把 $m_1$ 和 $m_2$ 的加速度表达式代入：

$$
\ddot{r} = \frac{F(r)}{m_2} - \frac{-F(r)}{m_1}
$$

化简得到：

$$
\ddot{r} = F(r)\left(\frac{1}{m_1} + \frac{1}{m_2}\right)
$$

上述等式正好符合牛顿第二定律的形式，所以我们可以把 $m_1$ 和 $m_2$ 合并成一个等效质量 $\mu$，称为折合质量 (Reduced Mass)，它的定义便是：

$$
\frac{1}{\mu} = \frac{1}{m_1} + \frac{1}{m_2}
$$

或

$$
\mu = \frac{m_1 m_2}{m_1 + m_2}
$$

### Velocity Verlet 算法

Velocity Verlet 算法是一种数值积分方法，用于求解分子动力学中的运动方程。它结合了位置和速度的更新，比简单的欧拉方法更稳定。

根据当前的坐标 $r(t)$ 、速度 $v(t)$ 和加速度 $a(t)$，直接算出下一帧的准确位置。这里用到了完整的匀加速位移公式（带着 $\frac{1}{2}at^2$）。

$$
r(t+\Delta t) = r(t) + v(t)\Delta t + \frac{1}{2}a(t)\Delta t^2
$$

（这一步走完后，原子已经来到了新位置，但此时它的速度还没更新。）

调用莫尔斯势的力函数 $F(r)$，计算新位置下的加速度。

$$
a(t+\Delta t) = \frac{F(r(t+\Delta t))}{\mu}
$$

现在，我们既有“刚才的加速度 $a(t)$”和“现在的加速度 $a(t+\Delta t)$”，在这短短的 $\Delta t$ 里，力的变化太剧烈了，怎么算速度最公平？

Velocity Verlet 取这两个加速度的平均值：

$$
v(t+\Delta t) = v(t) + \frac{1}{2}(a(t) + a(t+\Delta t))\Delta t
$$


该算法最主要的优势在于，系统总能量在长时间模拟中几乎不发生飘移。

## 建模



In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.animation as animation
from IPython.display import HTML, display
import pint

ureg = pint.UnitRegistry()
Q_ = ureg.Quantity

# ==========================================
# 1. 实验控制台 & 单位转换
# ==========================================
Ek_1 = Q_(2.0, 'eV'); Ek_2 = Q_(5.0, 'eV')
total_time = Q_(20.0, 'fs'); dt = Q_(0.01, 'fs')

De = Q_(4.75, 'eV'); re = Q_(0.74, 'angstrom'); a_param = Q_(1.93, 'angstrom**-1'); mu = Q_(0.504, 'amu')

eV_to_J = (1 * ureg.eV).to('J').magnitude
A_to_m = (1 * ureg.angstrom).to('m').magnitude
fs_to_s = (1 * ureg.fs).to('s').magnitude
amu_to_kg = (1 * ureg.amu).to('kg').magnitude

Ek_J_1 = Ek_1.magnitude * eV_to_J; Ek_J_2 = Ek_2.magnitude * eV_to_J
De_J = De.magnitude * eV_to_J; re_m = re.magnitude * A_to_m
a_m = a_param.magnitude / A_to_m; mu_kg = mu.magnitude * amu_to_kg
dt_s = dt.magnitude * fs_to_s; total_time_s = total_time.magnitude * fs_to_s
steps = int(total_time_s / dt_s)

# ==========================================
# 2. 物理引擎定义与循环
# ==========================================
def calc_force(r):
    exp_term = np.exp(-a_m * (r - re_m))
    return -2 * a_m * De_J * (1 - exp_term) * exp_term

def run_simulation(Ek_J):
    """独立的动力学计算模块"""
    r_list = np.zeros(steps); v_list = np.zeros(steps); t_list = np.zeros(steps)
    r_list[0] = re_m
    v_list[0] = -np.sqrt(2 * Ek_J / mu_kg) 
    a_curr = calc_force(r_list[0]) / mu_kg
    
    for i in range(steps - 1):
        r_new = r_list[i] + v_list[i] * dt_s + 0.5 * a_curr * dt_s**2
        a_new = calc_force(r_new) / mu_kg
        v_new = v_list[i] + 0.5 * (a_curr + a_new) * dt_s
        r_list[i+1] = r_new; v_list[i+1] = v_new; t_list[i+1] = t_list[i] + dt_s
        a_curr = a_new
    
    return r_list / A_to_m, t_list / fs_to_s

print("正在计算两条轨迹...")
r_A_1, t_fs_1 = run_simulation(Ek_J_1)
r_A_2, t_fs_2 = run_simulation(Ek_J_2)

def calc_potential_eV(r_A):
    exp_term = np.exp(-a_m * (r_A * A_to_m - re_m))
    return (De_J * (1 - exp_term)**2) / eV_to_J

# ==========================================
# 3. 动画生成器 (解耦输出，应用全新配色)
# ==========================================
def create_html_animation(r_A, t_fs, Ek_val, color_theme, title_text):
    """将单个模拟结果渲染为带 HTML 控制条的动画"""
    fig, ax = plt.subplots(figsize=(10, 4.5))
    
    r_bg = np.linspace(0.3, 4.5, 500)
    V_bg = calc_potential_eV(r_bg)
    ax.plot(r_bg, V_bg, color='blue', linewidth=2, alpha=0.5) # 稍微调浅蓝色背景势能井
    
    # 🎯 修改点 1：将解离能和平衡键长辅助线改为浅灰色
    ax.axhline(y=4.75, color='darkgray', linestyle='--', alpha=0.7)
    ax.axvline(x=0.74, color='darkgray', linestyle='--', alpha=0.7)
    
    ax.set_xlim(0.2, 4.0); ax.set_ylim(-0.5, 8.0)
    ax.set_title(title_text, fontsize=14, color=color_theme)
    ax.set_xlabel('Internuclear Distance $r$ (Å)', fontsize=12)
    ax.set_ylabel('Potential $V(r)$ (eV)', fontsize=12)
    ax.grid(True, linestyle=':', alpha=0.6)
    
    # 🎯 修改点 2：动点和彗星尾巴颜色将跟随传入的 color_theme 变量
    point, = ax.plot([], [], marker='o', linestyle='', markersize=10, zorder=5, color=color_theme)
    trail, = ax.plot([], [], linestyle='-', linewidth=2, alpha=0.6, zorder=4, color=color_theme)
    
    time_text = ax.text(0.05, 0.95, '', transform=ax.transAxes, fontsize=12,
                        verticalalignment='top', bbox=dict(boxstyle='round', facecolor='white', alpha=0.8))

    def animate(i):
        frame = min(i * 20, steps - 1)
        curr_r = r_A[frame]
        curr_V = calc_potential_eV(curr_r)
        
        point.set_data([curr_r], [curr_V])
        start_idx = max(0, frame - 500)
        trail.set_data(r_A[start_idx:frame], calc_potential_eV(r_A[start_idx:frame]))
        time_text.set_text(f'Time: {t_fs[frame]:.2f} fs\nDistance: {curr_r:.3f} Å')
        return point, trail, time_text

    ani = animation.FuncAnimation(fig, animate, frames=int(steps/20), interval=40, blit=True)
    plt.close(fig) 
    return HTML(ani.to_jshtml())

# 🎯 修改点 3：分别传入 'green' (绿色) 和 'red' (红色)
print("渲染完成！请查看下方独立的动画播放器：")
display(create_html_animation(r_A_1, t_fs_1, Ek_1.magnitude, 'green', 'Control Group: 2.0 eV (Bound State Oscillation)'))
display(create_html_animation(r_A_2, t_fs_2, Ek_2.magnitude, 'red', 'Experimental Group: 5.0 eV (Bond Dissociation)'))

## 结论

本实验基于一维 Velocity Verlet 分子动力学模拟，直观且严密地证实了：无论分子内部振动得初始相位（即原子相互靠近或远离）如何，只要系统吸收的总能量（$E_k$）超过化学键的解离能（$D_e$），最终必然导致化学键的断裂。

**核心现象与物理机制解析**：

1. **热能与微观动能的等效性**：宏观上的“吸热”过程，在微观尺度上直接转化为原子沿核间轴相对运动的振动动能。本实验向系统注入的初始动能（$2.0\text{ eV}$ 与 $5.0\text{ eV}$）即代表分子吸收的热量。

2. **莫尔斯势能井的极度不对称性（拓扑决定命运）**：观察背景势能曲线可知，化学键的势能分布并非对称的抛物线。其左侧（$r < r_e$）受泡利不相容原理和核排斥力主导，表现为几乎垂直的无限高斥力壁；其右侧（$r > r_e$）受长程吸引力主导，势能逐渐平缓并收敛于有限的解离能阈值（浅灰色水平虚线 $D_e = 4.75\text{ eV}$）。

3. **“向左冲刺”的绝处逢生（斥力壁反弹）**：实验专门测试了最极端的初始条件——即吸收能量后，原子具备极大的向左（相互压缩）的初速度。动图显示，系统并不会在左侧“撞毁”或卡死。极度陡峭的左侧斥力壁会在几飞秒内将原子的动能完全转化为势能，并以极大的反向加速度将其弹性反弹。这证明了：在无限高的左侧高墙面前，能量的唯一宣泄出口只能是右侧。

4. **能量阈值决定最终状态（对照组分析）**：
    - 对照组（绿色轨迹，$E_k = 2.0\text{ eV} < D_e$）： 虽然经历了剧烈的反弹，但由于总能量未超过解离能红线，动点在右侧引力坡上动能耗尽，最终被拉回，陷入束缚态下的不对称振荡（表现为未断键的高能激发态）。
    - 实验组（红色轨迹，$E_k = 5.0\text{ eV} > D_e$）： 动点从左侧绝壁反弹后，携带着巨大的残余动能冲向右侧。因总能量高于解离势垒，动点直接冲破 $D_e$ 虚线，原子间距离 $r \to \infty$，系统彻底脱离引力束缚（表现为宏观上的断键吸热现象）。

**最终总结**：

宏观层面的“断键吸热”规律，其微观本质是由**原子间相互作用势的不对称性**所决定的。吸收的热能打破了势能井内的能量封锁，而左侧无限高的核斥力壁则保证了这股冲破封锁的能量最终必定指向分离的方向（向右解离）。